# MultiVI set up

In [1]:
here::i_am("rna/trajectories/infer_trajectory.R")

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))


# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code



In [2]:
args <- list()
args$sce <-file.path(io$basedir,"data/processed/atac/archR/Matrices/PeakMatrix_summarized_experiment.rds")
args$metadataRNA <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$metadataATAC <- file.path(io$basedir, '/results/atac/archR/qc/sample_metadata_after_qc.txt.gz')
args$trajectory_name <- "epiblast_blood"
args$celltype_label <- "celltype"
args$outdir <- file.path(io$basedir,"results/epiblast_blood")

args$batch_variable = 'sample'
args$features = 3000
args$n_pcs <- 25
args$sample <- c('E7.5_rep1','E7.5_rep2','E7.75_rep1','E8.0_rep1','E8.0_rep2','E8.5_rep1','E8.5_rep2','E8.75_rep1','E8.75_rep2')
## END TEST ##

# I/O
dir.create(args$outdir, showWarnings=F, recursive=T)

# Trajectory 
opts$celltypes = c("Epiblast",
                    "Primitive_Streak" ,
                    "Nascent_mesoderm",
                    "ExE_mesoderm",
                    "Mixed_mesoderm",
                    "Allantois",
                    "Mesenchyme",
                    "Haematoendothelial_progenitors",
                    "Endothelium",
                    "Blood_progenitors_1",
                    "Blood_progenitors_2",
                    "Erythroid1",
                    "Erythroid2",
                    "Erythroid3")

In [3]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadataRNA) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE] %>%
    merge(.,fread(args$metadataATAC)[,c('cell', 'pass_atacQC')], by='cell') %>%
    .[pass_atacQC==TRUE]

stopifnot(args$celltype_label%in%colnames(sample_metadata))
sample_metadata <- sample_metadata   %>%
  .[,celltype:=eval(as.name(args$celltype_label))] %>%
  .[celltype%in%opts$celltypes] %>%
  .[,celltype:=factor(celltype,levels=opts$celltypes)] %>%
  .[sample%in% args$sample]

table(sample_metadata$celltype)


                      Epiblast               Primitive_Streak 
                           984                            542 
              Nascent_mesoderm                   ExE_mesoderm 
                          1327                           1173 
                Mixed_mesoderm                      Allantois 
                           265                            653 
                    Mesenchyme Haematoendothelial_progenitors 
                          3123                            913 
                   Endothelium            Blood_progenitors_1 
                           666                            230 
           Blood_progenitors_2                     Erythroid1 
                           565                           1119 
                    Erythroid2                     Erythroid3 
                           939                            187 

In [4]:
#############################
## Load ATAC accessibility ##
#############################

args$outdir = file.path(io$basedir, 'results/epiblast_blood/atac/')
features = fread(sprintf("%s/atac_variable_features.txt.gz",args$outdir))

sce = readRDS(args$sce)
sce = sce[,sample_metadata$cell]
sce = as(sce, 'SingleCellExperiment')

sce = sce[features$feature,]

assay(sce, 'counts') = assay(sce, 'PeakMatrix')
assay(sce, 'PeakMatrix') = NULL

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

sce_atac = sce

In [5]:
#########################
## Load RNA expression ##
#########################
args$sce <-file.path(io$basedir,"data/processed/rna/SingleCellExperiment.rds")

sce <- load_SingleCellExperiment(
  file = args$sce, 
  normalise = FALSE, 
  cells = sample_metadata$cell, 
  remove_non_expressed_genes = TRUE
)

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

sce_rna = sce

In [6]:
# Filter variable genes
gene_stats = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/results/gene_statistics/gene_statistics.txt.gz') %>%
    .[gene %in% rownames(sce_rna)] %>%
    .[order(-var_single_cells)]

sce_rna = sce_rna[head(gene_stats$gene, 2500),]

In [7]:
###################
## Combined SCEs ##
###################
sce = rbind(sce_rna, sce_atac)

In [8]:
# Anndata with only top 45k variable features
args$outdir <- file.path(io$basedir,"results/epiblast_blood")

sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata_multiVI.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:genotype, pass_rnaQC, doublet_call, pass_atacQC”


AnnData object with n_obs × n_vars = 12686 × 32500
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'stage', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell'
    var: 'idx'

In [9]:
var.dt = data.table('ID' = c(rownames(sce_rna), rownames(sce_atac)),
                    'modality' = c(rep('Gene Expression' , length(sce_rna)), rep('Peaks', length(sce_atac))))

In [10]:
fwrite(var.dt, sprintf("%s/variable.csv",args$outdir), sep=',')

# Export full adata with all features

In [4]:
#############################
## Load ATAC accessibility ##
#############################

args$outdir = file.path(io$basedir, 'results/epiblast_blood/atac/')

sce = readRDS(args$sce)
sce = sce[,sample_metadata$cell]
sce = as(sce, 'SingleCellExperiment')


assay(sce, 'counts') = assay(sce, 'PeakMatrix')
assay(sce, 'PeakMatrix') = NULL

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

sce_atac = sce

In [5]:
#########################
## Load RNA expression ##
#########################
args$sce <-file.path(io$basedir,"data/processed/rna/SingleCellExperiment.rds")

sce <- load_SingleCellExperiment(
  file = args$sce, 
  normalise = FALSE, 
  cells = sample_metadata$cell, 
  remove_non_expressed_genes = TRUE
)

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

sce_rna = sce

In [6]:
###################
## Combined SCEs ##
###################
sce = rbind(sce_rna, sce_atac)

In [7]:
# Anndata with only top 45k variable features
args$outdir <- file.path(io$basedir,"results/epiblast_blood")

sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata_multiVI_full.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:genotype, pass_rnaQC, doublet_call, pass_atacQC”


AnnData object with n_obs × n_vars = 12686 × 212770
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'stage', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell'
    var: 'idx'